In [1]:
from functions.analyte import ANALYTES
from functions.benchmark import benchmark_model
from networks.feed_forward_base import NeuralModel as FCBasic
from networks.feed_forward_tabular import NeuralModel as FCProteins
from networks.feed_forward_more_features import NeuralModel as FCAllFeatures
from networks.feed_forward_more_features_larger import NeuralModel as FCLarge
from networks.cnn_network import CNNModel as CNNBasic
from networks.cnn_network_small_kernels import CNNModel as CNNSmallKernels
from networks.cnn_network_large_kernels import CNNModel as CNNLargeKernels
from networks.cnn_global_maxpooling import CNNModel as CNNGlobalMaxPooling
from networks.cnn_no_global_pooling import CNNModel as CNNNoGlobalPooling
from networks.cnn_tiny import CNNModel as CNNTiny
from networks.cnn_small import CNNModel as CNNSmall
from networks.cnn_medium import CNNModel as CNNMedium
from networks.cnn_huge import CNNModel as CNNHuge
from networks.cnn_no_local_pooling import CNNModel as CNNNoLocalPooling
from networks.cnn_batchnorm1d import CNNModel as CNNBatchNorm1d
from networks.cnn_batchnorm_and_local_pool import CNNModel as CNNBatchAndLocalPool
from networks.cnn_shallow import CNNModel as CNNShallow
from networks.cnn_deep import CNNModel as CNNDeep
from networks.cnn_deeper import CNNModel as CNNDeeper
from networks.cnn_deepest import CNNModel as CNNDeepest
from networks.cnn_dilated_final_version import CNNModel as CNNDilations
import pandas as pd
import duckdb


FCBasicModel = FCBasic()
FCProteinsModel = FCProteins()
FCAllFeaturesModel = FCAllFeatures()
FClargeModel = FCLarge()
CNNBasicModel = CNNBasic()
CNNSmallkernelsModel = CNNSmallKernels()
CNNLargeKernelsModel = CNNLargeKernels()
CNNGlobalMaxPoolingModel = CNNGlobalMaxPooling()
CNNNoGlobalPoolingModel = CNNNoGlobalPooling()
CNNTinyModel = CNNTiny()
CNNSmallModel = CNNSmall()
CNNMediumModel = CNNMedium()
CNNHugeModel = CNNHuge()
CNNNoLocalPoolingModel = CNNNoLocalPooling()
CNNBatchNorm1dModel = CNNBatchNorm1d()
CNNBatchAndLocalPoolModel = CNNBatchAndLocalPool()
CNNShallowModel = CNNShallow()
CNNDeepModel = CNNDeep()
CNNDeeperModel = CNNDeeper()
CNNDeepestModel = CNNDeepest()
CNNDilationsModel = CNNDilations()

models = [
   FCBasicModel,
   FCProteinsModel,
   FCAllFeaturesModel,
   FClargeModel,
   CNNBasicModel,
   CNNSmallkernelsModel,
   CNNLargeKernelsModel,
   CNNGlobalMaxPoolingModel,
   CNNNoGlobalPoolingModel,
   CNNTinyModel,
   CNNSmallModel,
   CNNMediumModel,
    CNNHugeModel,
    CNNNoLocalPoolingModel,
    CNNBatchNorm1dModel,
    CNNBatchAndLocalPoolModel,
    CNNShallowModel,
    CNNDeepModel,
    CNNDeeperModel,
    CNNDeepestModel,
    CNNDilationsModel
]

con = duckdb.connect('../capillary.db')
df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 WHERE observation_nr = 1
                 """).df()
con.close()


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

protein_cols = [a.col for a in ANALYTES[:8]]
train_rows = train_rows.dropna(subset=protein_cols)
val_rows = val_rows.dropna(subset=protein_cols)
test_rows = test_rows.dropna(subset=protein_cols)

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index

train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]
print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

summary = []
if val_rows['label'].isna().any():
        print("NAN detected!")

for model in models:

    print(f"\nBenchmarking {model.name}")
    train_dl,val_dl,_ = model.build_dataloaders(train_rows,val_rows,test_rows)
    result_df = benchmark_model(
        model,
        train_dl,
        val_dl,
        val_rows['label'].to_numpy(),
        n_runs=10    )
    

    result_df.to_csv(
        f"../results/{model.name}.csv",
        index=False
    )

    summary.append({
        "Model": model.name,
        "AUC_mean": result_df["auc"].mean(),
        "AUC_std": result_df["auc"].std(),
        "Accuracy": result_df["accuracy"].mean(),
        "Sensitivity": result_df["sensitivity"].mean(),
        "Specificity": result_df["specificity"].mean(),
    })

summary_df = pd.DataFrame(summary)

for _, row in summary_df.iterrows():
    print(
        f"{row['Model'].replace('_',' ')} & "
        f"${row['AUC_mean']:.3f} \\pm {row['AUC_std']:.3f}$ & "
        f"{100*row['Accuracy']:.2f}\\% & "
        f"{100*row['Sensitivity']:.2f}\\% & "
        f"{100*row['Specificity']:.2f}\\% \\\\"
    )

Total parameters: 249,922
Total parameters: 251,970
Total parameters: 256,578
Total parameters: 469,782
Total parameters: 82,146
Total parameters: 209,186
Total parameters: 98,530
Total parameters: 688,354
Total parameters: 15,710
Total parameters: 45,266
Total parameters: 172,130
Total parameters: 1,505,826
Total parameters: 632,930
Total parameters: 633,122
Total parameters: 172,322
Total parameters: 314,674
Total parameters: 320,194
Total parameters: 324,898
Total parameters: 330,626
Total parameters: 324,898
Antal utan m-komponent i träningsdatan: 17895
Antal med m-komponent i träningsdatan: 2553

Benchmarking feed_forward_basic
feed_forward_basic: 1/10
  -> ny bästa modell sparad till ../models/feed_forward_basic.pth
Epoch   0 | train: 0.5540 | val: 0.4040 | acc: 90.42% | AUC: 0.885  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_basic.pth
Epoch   1 | train: 0.3774 | val: 0.2690 | acc: 92.73% | AUC: 0.938  | LR: 0.001
  -> ny bästa modell sparad till ../models

In [3]:
from pathlib import Path
import pandas as pd

summary = []

for file in sorted(Path("../results").glob("*.csv")):
    df = pd.read_csv(file)

    summary.append({
        "Model": file.stem,
        "auc_mean": df["auc"].mean(),
        "auc_std": df["auc"].std(),

        "acc_mean": df["accuracy"].mean(),
        "acc_std": df["accuracy"].std(),

        "sens_mean": df["sensitivity"].mean(),
        "sens_std": df["sensitivity"].std(),

        "spec_mean": df["specificity"].mean(),
        "spec_std": df["specificity"].std(),
    })

summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values(
    by="auc_mean",
    ascending=False
).reset_index(drop=True)

for _, row in summary_df.iterrows():
    print(
        f"{row['Model'].replace('_',' ')} & "
        f"${row['auc_mean']:.3f} \\pm {row['auc_std']:.3f}$ & "
        f"${100*row['acc_mean']:.2f} \\pm {100*row['acc_std']:.2f}$\\% & "
        f"${100*row['sens_mean']:.2f} \\pm {100*row['sens_std']:.2f}$\\% & "
        f"${100*row['spec_mean']:.2f} \\pm {100*row['spec_std']:.2f}$\\% \\\\"
    )

cnn huge & $0.997 \pm 0.000$ & $98.32 \pm 0.51$\% & $96.69 \pm 1.13$\% & $98.39 \pm 0.57$\% \\
cnn no global pooling & $0.997 \pm 0.001$ & $97.71 \pm 1.51$\% & $97.12 \pm 0.76$\% & $97.73 \pm 1.59$\% \\
cnn deepest & $0.997 \pm 0.001$ & $98.09 \pm 1.04$\% & $96.98 \pm 1.26$\% & $98.14 \pm 1.11$\% \\
cnn medium & $0.997 \pm 0.001$ & $98.33 \pm 0.50$\% & $96.62 \pm 0.96$\% & $98.41 \pm 0.55$\% \\
cnn deep & $0.997 \pm 0.001$ & $98.31 \pm 0.68$\% & $96.76 \pm 1.53$\% & $98.38 \pm 0.76$\% \\
cnn batchnorm and local pooling & $0.997 \pm 0.001$ & $98.41 \pm 0.69$\% & $96.83 \pm 1.08$\% & $98.48 \pm 0.76$\% \\
cnn dilation & $0.997 \pm 0.001$ & $98.24 \pm 0.97$\% & $96.33 \pm 2.21$\% & $98.33 \pm 1.09$\% \\
cnn batchnorm & $0.997 \pm 0.001$ & $98.46 \pm 0.50$\% & $95.76 \pm 2.02$\% & $98.58 \pm 0.57$\% \\
cnn deeper & $0.997 \pm 0.001$ & $98.62 \pm 0.65$\% & $96.69 \pm 1.03$\% & $98.71 \pm 0.71$\% \\
cnn no local pooling & $0.996 \pm 0.001$ & $98.30 \pm 0.52$\% & $95.54 \pm 1.58$\% & $98.42 \